## Тест LLM-кандидатів — v3: MamayLM проти Mistral, узгоджено зі схемою БД

**Що змінилось відносно v2 (і чому):**

1. **Схема задач тепер відображає реальну архітектуру бази** (architecture-proposal.md,
   розділ 2.4): статус — це не плаский enum, а пара `dimension` + `value`, як у
   реальній таблиці фактів ("вимір" + "значення": об'єкт №42, вимір "відпустка",
   значення "активна"). Параметр `unit` у Задачі В тепер **число** (`unit_number`),
   а не текстовий опис "3 батальйон" — бо це йде в типізовану колонку БД, де числовий
   ідентифікатор підрозділу доречніший за вільний текст.
2. **Звання тепер контрольований словник (enum), а не вільний текст** — це
   структурно усуває проблему, яка повторювалась у ВСІХ попередніх раундах тесту
   (Qwen на OpenRouter, Mistral на OpenRouter, Qwen й MamayLM локально): модель не
   могла стабільно привести звання+прізвище до називного відмінка без зайвих слів.
   Прізвище лишається вільним текстом (відкритий словник імен), але тепер без
   вимоги змінювати відмінок — копіюється як є.
3. **JSON-схема, а не текстова інструкція "поверни лише JSON"** — за перевіреними
   практиками prompt engineering 2025-2026 (grammar-constrained / structured
   outputs дають >99% дотримання схеми проти текстової інструкції). Локально це
   реалізовано через `llama-cpp-python`'s `response_format={"type": "json_object",
   "schema": ...}`, що конвертується в GBNF-граматику й фізично обмежує вибір
   токенів моделі — гарантований валідний JSON з правильними типами (це також
   структурно усуває баг MamayLM з попереднього раунду, де `service_related`
   повертався як рядок `"ТАК"` замість булевого `true`: схема з `"type": "boolean"`
   робить таку відповідь неможливою на рівні семплінгу, не лише бажаною).
4. **Промпти переписані**: system/user розділення, XML-теги для структури
   (`<rules>`, `<example>`, `<document>`), стислі — оскільки формат тепер
   гарантує граматика, а не текст, велика частина "не пиши markdown, поверни
   лише JSON" мови просто не потрібна.

**Моделі:** `MamayLM-Gemma-3-12B-IT-v2.0-Q4_K_M.gguf` (Gemma Terms of Use, 7.3 GB) проти
`Mistral-Small-24B-Instruct-2501-Q4_K_M.gguf` (Apache 2.0, 14.33 GB). Послідовне
завантаження (одна модель повністю відпрацьовує, вивантажується, тоді наступна) —
обов'язково після інциденту з нестачею RAM у попередньому раунді.

**N_REPEATS=1** — локальний інференс детермінований за конструкцією.

**Джерела practices, використаних тут:**
- Structured outputs / JSON schema constrained decoding: https://agenta.ai/blog/the-guide-to-structured-outputs-and-function-calling-with-llms
- llama-cpp-python response_format -> GBNF grammar: https://github.com/abetlen/llama-cpp-python/discussions/1173
- Anthropic prompt engineering guide (clarity, XML tags, examples): https://docs.claude.com (via огляд https://scalablehuman.com/2025/07/02/review-anthropics-prompt-engineering-guide/)

In [1]:
import re, json, statistics, time, gc
from pathlib import Path
from llama_cpp import Llama

MODELS_DIR = Path("../models")

MODEL_PATHS = {
    "MamayLM-Gemma-3-12B-v2.0": MODELS_DIR / "MamayLM-Gemma-3-12B-IT-v2.0-Q4_K_M.gguf",
    "Mistral-Small-24B-Instruct-2501": MODELS_DIR / "Mistral-Small-24B-Instruct-2501-Q4_K_M.gguf",
}

N_REPEATS = 1

RANK_ENUM = [
    "солдат", "молодший сержант", "сержант", "старший сержант", "старшина",
    "прапорщик", "молодший лейтенант", "лейтенант", "старший лейтенант",
    "капітан", "майор", "підполковник", "полковник", None,
]

VACATION_CATEGORY_ENUM = ["annual_leave", "family_circumstances_leave", "health_leave", "other", None]

def normalize_value(v):
    if v is None:
        return None
    if isinstance(v, str):
        v = v.strip().lower()
        if v in ("", "null", "невідомо", "не вказано", "n/a", "none"):
            return None
        if re.fullmatch(r"-?\d+", v):
            return int(v)
        return v
    return v

def dict_key(d):
    return json.dumps(d, sort_keys=True, ensure_ascii=False) if isinstance(d, dict) else repr(d)


### Задача А — екстракція шумних полів (JSON-схема per-case, enum для звання/категорії відпустки)

In [2]:
SYSTEM_PROMPT_A = (
    "Ти - модуль екстракції структурованих даних із українських військових "
    "документів для системи, що зберігає результат у типізованих колонках "
    "бази даних PostgreSQL.\n\n"
    "<rules>\n"
    "1. Заповнюй лише ті поля, для яких є пряма textual підстава в документі. "
    "Немає підстави - null.\n"
    "2. Не обчислюй і не вигадуй значення (напр. не виводь дату закінчення з "
    "дати початку і тривалості, якщо вона не вказана прямо).\n"
    "3. Якщо замість реального значення в тексті стоїть заглушка/плейсхолдер "
    "(\"XXX\", \"_____\" тощо) - це так само null, не буквальне значення заглушки.\n"
    "4. Дати завжди у форматі DD.MM.YYYY.\n"
    "5. Прізвище копіюй як воно записано в тексті, без зміни відмінка - "
    "нормалізацію відмінків прізвищ система робить окремим детермінованим "
    "кроком, не твоя задача.\n"
    "6. Звання і категорію відпустки обирай лише зі списку допустимих значень "
    "у схемі відповіді - не вигадуй нових.\n"
    "</rules>\n\n"
    "<example>\n"
    "Текст: \"Прошу надати відрядження терміном на 5 діб з 12 березня 2024 "
    "року. Обов'язки чергового по частині покласти на капітана Іваненка. "
    "Контактний номер: XXX.\"\n"
    "Поля: duration_days, start_date, responsible_officer_rank, "
    "responsible_officer_surname, contact_phone\n"
    "Відповідь: {\"duration_days\": 5, \"start_date\": \"12.03.2024\", "
    "\"responsible_officer_rank\": \"капітан\", "
    "\"responsible_officer_surname\": \"Іваненка\", \"contact_phone\": null}\n"
    "</example>"
)

TASK_A_CASES = [
    {
        "label": "1.webp - заповнений рапорт (чистий скан)",
        "text": (
            "Прошу Вас надати мені частину щорічної основної відпустки за 2023 рік "
            "терміном на 10 діб з 01 січні 2023. Обов'язки помічника начальника штабу "
            "прошу покласти на офіцера штабу майора Коцюбу. Відпустку буду проводити за "
            "адресою: м. Вінниця, вул. Велика Бандерівська, б. 11 кв. 1155. "
            "Моб.телефон 067-891-23-45. У Збройних Силах України з 2014 року."
        ),
        "fields": ["vacation_category", "duration_days", "start_date", "end_date",
                   "phone", "replacement_officer_rank", "replacement_officer_surname"],
        "schema": {
            "type": "object",
            "properties": {
                "vacation_category": {"type": ["string", "null"], "enum": VACATION_CATEGORY_ENUM},
                "duration_days": {"type": ["integer", "null"]},
                "start_date": {"type": ["string", "null"]},
                "end_date": {"type": ["string", "null"]},
                "phone": {"type": ["string", "null"]},
                "replacement_officer_rank": {"type": ["string", "null"], "enum": RANK_ENUM},
                "replacement_officer_surname": {"type": ["string", "null"]},
            },
            "required": ["vacation_category", "duration_days", "start_date", "end_date",
                         "phone", "replacement_officer_rank", "replacement_officer_surname"],
        },
        "golden": {
            "vacation_category": "annual_leave",
            "duration_days": 10,
            "start_date": "01.01.2023",
            "end_date": None,
            "phone": "067-891-23-45",
            "replacement_officer_rank": "майор",
            "replacement_officer_surname": "Коцюбу",
        },
    },
    {
        "label": "довідка ВЛК (синтетична, без приватних даних)",
        "text": (
            "ДОВІДКА військово-лікарської комісії. Проведено медичний огляд ВЛК "
            "клінічна лікарня м. Києва. Діагноз та постанова ВЛК про причинний зв'язок "
            "захворювання: Стан після мінно-вибухової травми (23.06.2023 р.). За наказом "
            "МОЗ від 04.07.2007 № 370 травма легка. Травма, ТАК, пов'язана з проходженням "
            "військової служби (довідка про обставини травми не надана). Потребує "
            "відпустки за станом здоров'я на 30 (тридцять) календарних днів."
        ),
        "fields": ["injury_date", "vacation_days", "service_related",
                   "circumstances_certificate_provided", "rank"],
        "schema": {
            "type": "object",
            "properties": {
                "injury_date": {"type": ["string", "null"]},
                "vacation_days": {"type": ["integer", "null"]},
                "service_related": {"type": ["boolean", "null"]},
                "circumstances_certificate_provided": {"type": ["boolean", "null"]},
                "rank": {"type": ["string", "null"], "enum": RANK_ENUM},
            },
            "required": ["injury_date", "vacation_days", "service_related",
                         "circumstances_certificate_provided", "rank"],
        },
        "golden": {
            "injury_date": "23.06.2023",
            "vacation_days": 30,
            "service_related": True,
            "circumstances_certificate_provided": False,
            "rank": None,
        },
    },
    {
        "label": "public - порожній бланк (нічого не заповнено)",
        "text": (
            "Командиру В/ч _____. Рапорт. Прошу надати мені, _____ (звання, ПІБ), "
            "відпустку на _____ діб за сімейними обставинами. Відпустку буду проводити "
            "за адресою: _____. Тел.: _____."
        ),
        "fields": ["rank", "full_name", "duration_days", "address", "phone"],
        "schema": {
            "type": "object",
            "properties": {
                "rank": {"type": ["string", "null"], "enum": RANK_ENUM},
                "full_name": {"type": ["string", "null"]},
                "duration_days": {"type": ["integer", "null"]},
                "address": {"type": ["string", "null"]},
                "phone": {"type": ["string", "null"]},
            },
            "required": ["rank", "full_name", "duration_days", "address", "phone"],
        },
        "golden": {
            "rank": None, "full_name": None, "duration_days": None,
            "address": None, "phone": None,
        },
    },
    {
        "label": "raport-optimized - знеособлений шаблон (XXX замість значень)",
        "text": (
            "Прошу вас надати мені частину щорічної основної відпустки терміном на "
            "10 (десять) діб із 01.10.2023 року. Відпустку буду проводити за адресою: "
            "країна XXX, місто XXX, вул. XXX. Телефон для оповіщення: XXX."
        ),
        "fields": ["duration_days", "start_date", "country", "city", "phone"],
        "schema": {
            "type": "object",
            "properties": {
                "duration_days": {"type": ["integer", "null"]},
                "start_date": {"type": ["string", "null"]},
                "country": {"type": ["string", "null"]},
                "city": {"type": ["string", "null"]},
                "phone": {"type": ["string", "null"]},
            },
            "required": ["duration_days", "start_date", "country", "city", "phone"],
        },
        "golden": {
            "duration_days": 10, "start_date": "01.10.2023",
            "country": None, "city": None, "phone": None,
        },
    },
]

def build_task_a_messages(case):
    fields_str = ", ".join(case["fields"])
    user = f"<document>\n{case['text']}\n</document>\n\nВитягни поля: {fields_str}."
    return [
        {"role": "system", "content": SYSTEM_PROMPT_A},
        {"role": "user", "content": user},
    ]

def score_task_a(golden, predicted):
    if not isinstance(predicted, dict):
        return {"valid_json": False, "field_scores": {}, "accuracy": 0.0}
    field_scores = {}
    for field, gold_val in golden.items():
        pred_val = normalize_value(predicted.get(field))
        gold_norm = normalize_value(gold_val)
        field_scores[field] = (pred_val == gold_norm)
    accuracy = sum(field_scores.values()) / len(field_scores)
    return {"valid_json": True, "field_scores": field_scores, "accuracy": accuracy}


### Задача Б — резолюція терміну в довідник вимір+значення (dimension+value, як у реальній таблиці фактів)

In [3]:
DIMENSION_VALUE_VOCAB = """
- вимір=vacation, значення=active - особа перебуває у відпустці (будь-яке формулювання: "у відпустці", "у щорічній відпустці" тощо)
- вимір=deployment, значення=active - особа у відрядженні
- вимір=service, значення=discharged - особа звільнена з військової служби
- вимір=medical, значення=hospital_treatment - особа проходить лікування у шпиталі
""".strip()

SYSTEM_PROMPT_B = (
    "Ти визначаєш, якому виміру й значенню з контрольованого довідника "
    "відповідає термін із документа.\n\n"
    f"<vocabulary>\n{DIMENSION_VALUE_VOCAB}\n</vocabulary>\n\n"
    "Якщо термін НЕ відповідає точно жодному з варіантів довідника - постав "
    "recognized=false, dimension=null, value=null. Не підбирай найближчий за "
    "змістом варіант."
)

TASK_B_SCHEMA = {
    "type": "object",
    "properties": {
        "recognized": {"type": "boolean"},
        "dimension": {"type": ["string", "null"], "enum": ["vacation", "deployment", "service", "medical", None]},
        "value": {"type": ["string", "null"], "enum": ["active", "discharged", "hospital_treatment", None]},
    },
    "required": ["recognized", "dimension", "value"],
}

TASK_B_CASES = [
    {"term": "у відпустці", "golden": {"recognized": True, "dimension": "vacation", "value": "active"}},
    {"term": "перебуває у щорічній відпустці", "golden": {"recognized": True, "dimension": "vacation", "value": "active"}},
    {"term": "у відрядженні", "golden": {"recognized": True, "dimension": "deployment", "value": "active"}},
    {"term": "звільнений з військової служби", "golden": {"recognized": True, "dimension": "service", "value": "discharged"}},
    {"term": "проходить лікування у шпиталі", "golden": {"recognized": True, "dimension": "medical", "value": "hospital_treatment"}},
    {"term": "у самоволці", "golden": {"recognized": False, "dimension": None, "value": None}},
]

def build_task_b_messages(term):
    return [
        {"role": "system", "content": SYSTEM_PROMPT_B},
        {"role": "user", "content": f"<term>{term}</term>"},
    ]

def score_task_b(golden, predicted):
    if not isinstance(predicted, dict):
        return False
    return all(predicted.get(k) == v for k, v in golden.items())


### Задача В — вибір шаблону запиту + параметри (unit як число, dimension+value для статусу)

In [4]:
QUERY_TEMPLATES = """
- COUNT_BY_STATUS(dimension, value) - порахувати людей за виміром+значенням
- LIST_BY_STATUS(dimension, value) - перелічити людей за виміром+значенням
- COUNT_BY_STATUS_UNIT(dimension, value, unit_number) - порахувати у конкретному підрозділі
- COUNT_BY_STATUS_DATE_RANGE(dimension, value, date_from, date_to) - порахувати за період
- UNSUPPORTED - жоден шаблон не підходить
""".strip()

SYSTEM_PROMPT_C = (
    "Ти обираєш шаблон запиту до бази даних із фіксованого списку і заповнюєш "
    "параметри. Ти ніколи не пишеш SQL сам.\n\n"
    f"<templates>\n{QUERY_TEMPLATES}\n</templates>\n\n"
    f"<vocabulary>\n{DIMENSION_VALUE_VOCAB}\n</vocabulary>\n\n"
    "unit_number - це ЧИСЛО підрозділу (напр. \"3 батальйон\" -> unit_number=3), "
    "не текстовий опис: параметр іде напряму в типізовану цілочисельну колонку "
    "бази даних. Дати - у форматі DD.MM.YYYY. Якщо жоден шаблон не підходить - "
    "template=UNSUPPORTED, усі параметри null."
)

TASK_C_SCHEMA = {
    "type": "object",
    "properties": {
        "template": {
            "type": "string",
            "enum": ["COUNT_BY_STATUS", "LIST_BY_STATUS", "COUNT_BY_STATUS_UNIT",
                     "COUNT_BY_STATUS_DATE_RANGE", "UNSUPPORTED"],
        },
        "params": {
            "type": "object",
            "properties": {
                "dimension": {"type": ["string", "null"], "enum": ["vacation", "deployment", "service", "medical", None]},
                "value": {"type": ["string", "null"], "enum": ["active", "discharged", "hospital_treatment", None]},
                "unit_number": {"type": ["integer", "null"]},
                "date_from": {"type": ["string", "null"]},
                "date_to": {"type": ["string", "null"]},
            },
            "required": ["dimension", "value", "unit_number", "date_from", "date_to"],
        },
    },
    "required": ["template", "params"],
}

TASK_C_CASES = [
    {
        "question": "Скільки людей зараз у відпустці?",
        "golden": {"template": "COUNT_BY_STATUS",
                   "params": {"dimension": "vacation", "value": "active", "unit_number": None, "date_from": None, "date_to": None}},
    },
    {
        "question": "Хто зараз у відрядженні?",
        "golden": {"template": "LIST_BY_STATUS",
                   "params": {"dimension": "deployment", "value": "active", "unit_number": None, "date_from": None, "date_to": None}},
    },
    {
        "question": "Скільки людей у відпустці в 3 батальйоні?",
        "golden": {"template": "COUNT_BY_STATUS_UNIT",
                   "params": {"dimension": "vacation", "value": "active", "unit_number": 3, "date_from": None, "date_to": None}},
    },
    {
        "question": "Скільки було звільнено з 01.06.2023 по 31.08.2023?",
        "golden": {"template": "COUNT_BY_STATUS_DATE_RANGE",
                   "params": {"dimension": "service", "value": "discharged", "unit_number": None,
                              "date_from": "01.06.2023", "date_to": "31.08.2023"}},
    },
    {
        "question": "Яка погода в Києві сьогодні?",
        "golden": {"template": "UNSUPPORTED",
                   "params": {"dimension": None, "value": None, "unit_number": None, "date_from": None, "date_to": None}},
    },
]

def build_task_c_messages(question):
    return [
        {"role": "system", "content": SYSTEM_PROMPT_C},
        {"role": "user", "content": f"<question>{question}</question>"},
    ]

def score_task_c(golden, predicted):
    if not isinstance(predicted, dict):
        return False
    if predicted.get("template") != golden["template"]:
        return False
    pred_params = {k: normalize_value(v) for k, v in (predicted.get("params") or {}).items()}
    gold_params = {k: normalize_value(v) for k, v in golden["params"].items()}
    return pred_params == gold_params


### Прогін: одна модель повністю (усі 3 задачі, grammar-enforced JSON), потім вивантаження, потім наступна

In [5]:
def run_all_tasks_for_model(model_name: str, model_path) -> dict:
    print(f"=== Завантаження {model_name} ({model_path.stat().st_size / 1e9:.1f} GB) ===")
    t0 = time.time()
    llm = Llama(model_path=str(model_path), n_ctx=4096, n_threads=6, n_threads_batch=12, verbose=False)
    print(f"  завантажено за {time.time() - t0:.0f}с")

    def call(messages, schema) -> dict:
        t0 = time.time()
        resp = llm.create_chat_completion(
            messages=messages,
            temperature=0,
            response_format={"type": "json_object", "schema": schema},
        )
        print(f"    відповідь за {time.time() - t0:.0f}с")
        raw = resp["choices"][0]["message"]["content"]
        try:
            return json.loads(raw)
        except json.JSONDecodeError:
            return None

    task_a = []
    for case in TASK_A_CASES:
        parsed_list = [call(build_task_a_messages(case), case["schema"]) for _ in range(N_REPEATS)]
        scores = [score_task_a(case["golden"], p) for p in parsed_list]
        accuracies = [s["accuracy"] for s in scores]
        consistent = len({dict_key(p) for p in parsed_list}) == 1
        result = {
            "case": case["label"], "mean_accuracy": statistics.mean(accuracies),
            "min_accuracy": min(accuracies), "max_accuracy": max(accuracies),
            "consistent_across_repeats": consistent, "parsed_list": parsed_list,
            "field_scores_list": [s["field_scores"] for s in scores],
        }
        task_a.append(result)
        stability = "стабільно" if consistent else "НЕСТАБІЛЬНО"
        print(f"[{model_name}] Задача А / {case['label']}: точність={result['mean_accuracy']:.0%} — {stability}")

    task_b = []
    for case in TASK_B_CASES:
        parsed_list = [call(build_task_b_messages(case["term"]), TASK_B_SCHEMA) for _ in range(N_REPEATS)]
        correct_list = [score_task_b(case["golden"], p) for p in parsed_list]
        correct_rate = sum(correct_list) / len(correct_list)
        consistent = len({dict_key(p) for p in parsed_list}) == 1
        task_b.append({
            "term": case["term"], "golden": case["golden"], "parsed_list": parsed_list,
            "correct_rate": correct_rate, "consistent_across_repeats": consistent,
        })
        stability = "стабільно" if consistent else "НЕСТАБІЛЬНО"
        print(f"[{model_name}] Задача Б / \"{case['term']}\" -> {parsed_list} — {stability}")

    task_c = []
    for case in TASK_C_CASES:
        parsed_list = [call(build_task_c_messages(case["question"]), TASK_C_SCHEMA) for _ in range(N_REPEATS)]
        correct_list = [score_task_c(case["golden"], p) for p in parsed_list]
        correct_rate = sum(correct_list) / len(correct_list)
        consistent = len({dict_key(p) for p in parsed_list}) == 1
        task_c.append({
            "question": case["question"], "golden": case["golden"], "parsed_list": parsed_list,
            "correct_rate": correct_rate, "consistent_across_repeats": consistent,
        })
        stability = "стабільно" if consistent else "НЕСТАБІЛЬНО"
        print(f"[{model_name}] Задача В / \"{case['question']}\" -> {parsed_list} — {stability}")

    del llm
    gc.collect()
    print(f"=== {model_name} вивантажено з пам'яті ===\n")
    return {"task_a": task_a, "task_b": task_b, "task_c": task_c}

RESULTS = {}
for model_name, model_path in MODEL_PATHS.items():
    RESULTS[model_name] = run_all_tasks_for_model(model_name, model_path)


=== Завантаження MamayLM-Gemma-3-12B-v2.0 (7.3 GB) ===


llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)


  завантажено за 160с
    відповідь за 406с
[MamayLM-Gemma-3-12B-v2.0] Задача А / 1.webp - заповнений рапорт (чистий скан): точність=71% — стабільно
    відповідь за 46с
[MamayLM-Gemma-3-12B-v2.0] Задача А / довідка ВЛК (синтетична, без приватних даних): точність=100% — стабільно
    відповідь за 24с
[MamayLM-Gemma-3-12B-v2.0] Задача А / public - порожній бланк (нічого не заповнено): точність=100% — стабільно
    відповідь за 32с
[MamayLM-Gemma-3-12B-v2.0] Задача А / raport-optimized - знеособлений шаблон (XXX замість значень): точність=100% — стабільно
    відповідь за 32с
[MamayLM-Gemma-3-12B-v2.0] Задача Б / "у відпустці" -> [{'recognized': True, 'dimension': 'vacation', 'value': 'active'}] — стабільно
    відповідь за 10с
[MamayLM-Gemma-3-12B-v2.0] Задача Б / "перебуває у щорічній відпустці" -> [{'recognized': True, 'dimension': 'vacation', 'value': 'active'}] — стабільно
    відповідь за 9с
[MamayLM-Gemma-3-12B-v2.0] Задача Б / "у відрядженні" -> [{'recognized': True, 'dimension':

KeyboardInterrupt: 

### Підсумок

In [ ]:
print("=" * 70)
print("ПІДСУМОК (локально, grammar-enforced JSON-схема, N=1 повтор)")
print("=" * 70)
for model_name, r in RESULTS.items():
    a_acc = statistics.mean(x["mean_accuracy"] for x in r["task_a"])
    a_consistent = sum(x["consistent_across_repeats"] for x in r["task_a"])
    b_acc = statistics.mean(x["correct_rate"] for x in r["task_b"])
    b_consistent = sum(x["consistent_across_repeats"] for x in r["task_b"])
    c_acc = statistics.mean(x["correct_rate"] for x in r["task_c"])
    c_consistent = sum(x["consistent_across_repeats"] for x in r["task_c"])
    print(f"\n{model_name}:")
    print(f"  Задача А: точність={a_acc:.0%}, стабільно на {a_consistent}/{len(r['task_a'])} кейсів")
    print(f"  Задача Б: точність={b_acc:.0%}, стабільно на {b_consistent}/{len(r['task_b'])} кейсів")
    print(f"  Задача В: точність={c_acc:.0%}, стабільно на {c_consistent}/{len(r['task_c'])} кейсів")

print()
for model_name, r in RESULTS.items():
    for res in r["task_a"]:
        wrong = {f: rate for f, rate in
                 {fn: sum(1 for fs in res["field_scores_list"] if not fs.get(fn, False)) / len(res["field_scores_list"])
                  for fn in (res["field_scores_list"][0].keys() if res["field_scores_list"] else [])}.items()
                 if rate > 0}
        if wrong:
            print(f"[{model_name}] {res['case']}: помилки по полях {wrong}")
            print(f"  приклад: {res['parsed_list'][0]}")
